<a href="https://colab.research.google.com/github/Ashu-42/quant_research_crypto_volatility_forecasting/blob/quant_DL_ashu/notebooks/14_All_Models_Train_Validation_Test_Errors_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14 — Final Train / Validation / Test Error Comparison

This notebook performs the **final split-wise evaluation** of the six frozen forecasting models from the primary daily squared-return variance experiment.

## Models

### ARCH family
- GARCH(1,1)
- GJR-GARCH(1,1)
- EGARCH(1,1)

### Global deep learning
- MLP 30D
- GRU 7D
- LSTM 14D

## Target

All errors in this notebook use the original primary variance target:

\[
RV_t = r_t^2
\]

The DL models were trained on:

\[
\log(RV_t+\epsilon)
\]

but predictions are converted back to the variance scale before evaluation.

## Metrics

For every asset, model and split:

- MAE
- RMSE
- QLIKE

are calculated on the **variance scale**.

## Split definition

- Train target dates: through `2024-07-31`
- Validation: `2024-08-01` to `2025-07-31`
- Test: `2025-08-01` to `2026-07-31`

Validation and test each contain 365 target dates.

### Fair train comparison

The selected DL models use different lookbacks. Their earliest available training target can therefore differ.

For the final train-error table, this notebook uses the **intersection of training target dates available to MLP-30D, GRU-7D and LSTM-14D within each asset**. ARCH training predictions are restricted to exactly the same dates.

This makes train errors directly comparable across all six models.

## Important final-test rule

Running this notebook opens the previously untouched test period.

After the test results are viewed, **do not use the test results for any further model selection, lookback selection, architecture tuning or hyperparameter tuning**. They should be treated as the final held-out performance estimate.

## 1. Imports and project paths

In [1]:
from google.colab import drive
from pathlib import Path

import importlib.util
import json
import subprocess
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

if importlib.util.find_spec("arch") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "arch",
        ]
    )

from arch import arch_model

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Quant Research"
)

DAILY_FEATURE_PATH = (
    PROJECT_DIR
    / "data"
    / "model_ready"
    / "crypto_daily_features_v1.parquet"
)

GLOBAL_ARTIFACT_ROOT = (
    PROJECT_DIR
    / "data"
    / "model_ready"
    / "GLOBAL_4_ASSETS"
)

GLOBAL_MODEL_ROOT = (
    PROJECT_DIR
    / "models"
    / "GLOBAL_4_ASSETS"
)

GLOBAL_RESULT_ROOT = (
    PROJECT_DIR
    / "results"
    / "GLOBAL_4_ASSETS"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "results"
    / "FINAL_TRAIN_VALIDATION_TEST_COMPARISON"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Daily features:", DAILY_FEATURE_PATH)
print("Global artifacts:", GLOBAL_ARTIFACT_ROOT)
print("Global models:", GLOBAL_MODEL_ROOT)
print("Output:", OUTPUT_DIR)

Mounted at /content/drive
Daily features: /content/drive/MyDrive/Quant Research/data/model_ready/crypto_daily_features_v1.parquet
Global artifacts: /content/drive/MyDrive/Quant Research/data/model_ready/GLOBAL_4_ASSETS
Global models: /content/drive/MyDrive/Quant Research/models/GLOBAL_4_ASSETS
Output: /content/drive/MyDrive/Quant Research/results/FINAL_TRAIN_VALIDATION_TEST_COMPARISON


## 2. Frozen experiment configuration

In [2]:
ASSETS = [
    "BTCUSDT",
    "ETHUSDT",
    "SOLUSDT",
    "XRPUSDT",
]

ASSET_LABELS = {
    "BTCUSDT": "BTC",
    "ETHUSDT": "ETH",
    "SOLUSDT": "SOL",
    "XRPUSDT": "XRP",
}

ASSET_ORDER = [
    "BTC",
    "ETH",
    "SOL",
    "XRP",
]

MODEL_LOOKBACKS = {
    "MLP": 30,
    "GRU": 7,
    "LSTM": 14,
}

FEATURE_COLUMNS = [
    "log_realized_variance",
    "daily_return",
    "absolute_daily_return",
    "high_low_range",
    "log_volume",
    "rv_mean_7d",
    "rv_mean_30d",
]

TARGET_COLUMN = "target_log_rv"
ACTUAL_VARIANCE_COLUMN = "target_rv"

TRAIN_END = pd.Timestamp(
    "2024-07-31",
    tz="UTC",
)

VALIDATION_START = pd.Timestamp(
    "2024-08-01",
    tz="UTC",
)

VALIDATION_END = pd.Timestamp(
    "2025-07-31",
    tz="UTC",
)

TEST_START = pd.Timestamp(
    "2025-08-01",
    tz="UTC",
)

TEST_END = pd.Timestamp(
    "2026-07-31",
    tz="UTC",
)

RETURN_SCALE = 100.0
EPSILON = 1e-12
MIN_TRAIN_OBS = 100

ARCH_SPECS = {
    "GARCH(1,1)": {
        "vol": "GARCH",
        "p": 1,
        "o": 0,
        "q": 1,
    },
    "GJR-GARCH(1,1)": {
        "vol": "GARCH",
        "p": 1,
        "o": 1,
        "q": 1,
    },
    "EGARCH(1,1)": {
        "vol": "EGARCH",
        "p": 1,
        "o": 1,
        "q": 1,
    },
}

DISPLAY_MODEL_MAP = {
    "GARCH(1,1)": "GARCH (1,1)",
    "GJR-GARCH(1,1)": "GJR-GARCH (1,1)",
    "EGARCH(1,1)": "EGARCH (1,1)",
    "Global-MLP-30d": "MLP 30D",
    "Global-GRU-7d": "GRU 7D",
    "Global-LSTM-14d": "LSTM 14D",
}

MODEL_ORDER = [
    "GARCH (1,1)",
    "GJR-GARCH (1,1)",
    "EGARCH (1,1)",
    "MLP 30D",
    "GRU 7D",
    "LSTM 14D",
]

SPLIT_ORDER = [
    "Train",
    "Validation",
    "Test",
]

## 3. Load and QC the canonical primary dataset

In [3]:
if not DAILY_FEATURE_PATH.exists():
    raise FileNotFoundError(
        "Canonical primary feature file not found:\n"
        f"{DAILY_FEATURE_PATH}"
    )

daily_df = pd.read_parquet(
    DAILY_FEATURE_PATH
)

required_columns = {
    "symbol",
    "date",
    "target_date",
    "split",
    TARGET_COLUMN,
    ACTUAL_VARIANCE_COLUMN,
    *FEATURE_COLUMNS,
}

missing_columns = (
    required_columns
    - set(daily_df.columns)
)

if missing_columns:
    raise ValueError(
        "Canonical daily feature file "
        f"is missing: {missing_columns}"
    )

daily_df["date"] = pd.to_datetime(
    daily_df["date"],
    utc=True,
    errors="raise",
)

daily_df["target_date"] = pd.to_datetime(
    daily_df["target_date"],
    utc=True,
    errors="coerce",
)

daily_df = (
    daily_df.loc[
        daily_df["symbol"].isin(
            ASSETS
        )
    ]
    .sort_values(
        [
            "symbol",
            "date",
        ]
    )
    .reset_index(drop=True)
)

if daily_df.duplicated(
    subset=[
        "symbol",
        "date",
    ]
).any():
    raise ValueError(
        "Duplicate symbol-date rows found."
    )

split_qc = (
    daily_df.loc[
        daily_df["split"].isin(
            [
                "train",
                "validation",
                "test",
            ]
        )
        & daily_df[
            ACTUAL_VARIANCE_COLUMN
        ].notna()
    ]
    .groupby("split")
    .agg(
        first_target=(
            "target_date",
            "min",
        ),
        last_target=(
            "target_date",
            "max",
        ),
        rows=(
            "target_date",
            "size",
        ),
    )
)

display(split_qc)

assert (
    split_qc.loc[
        "validation",
        "first_target",
    ]
    == VALIDATION_START
)

assert (
    split_qc.loc[
        "validation",
        "last_target",
    ]
    == VALIDATION_END
)

assert (
    split_qc.loc[
        "test",
        "first_target",
    ]
    == TEST_START
)

assert (
    split_qc.loc[
        "test",
        "last_target",
    ]
    == TEST_END
)

print("Canonical dataset QC passed.")

,first_target,last_target,rows
split,,,
test,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00,1460
train,2017-08-18 00:00:00+00:00,2024-07-31 00:00:00+00:00,8810
validation,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00,1460


Canonical dataset QC passed.


## 4. Reconstruct the exact sequence metadata used by the Global DL pipeline

The original global notebook saved the pooled train / validation / test arrays and per-asset target scalers.

Train metadata was not required by the original validation notebook, so this notebook reconstructs the sequence metadata using the **same sequence-generation logic** and then validates its alignment against the saved pooled arrays.

No feature scaling or model training is repeated here.

In [4]:
def create_asset_sequence_metadata(
    asset_df,
    feature_columns,
    lookback_days,
):
    asset_df = (
        asset_df
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )

    metadata_rows = []

    for end_index in range(
        lookback_days - 1,
        len(asset_df),
    ):
        start_index = (
            end_index
            - lookback_days
            + 1
        )

        window = asset_df.iloc[
            start_index:
            end_index + 1
        ]

        final_row = (
            asset_df.iloc[
                end_index
            ]
        )

        dates = window["date"]

        expected_dates = pd.date_range(
            start=dates.iloc[0],
            periods=lookback_days,
            freq="D",
            tz=dates.iloc[0].tz,
        )

        if not np.array_equal(
            dates.to_numpy(),
            expected_dates.to_numpy(),
        ):
            continue

        if (
            window[
                feature_columns
            ]
            .isna()
            .any()
            .any()
        ):
            continue

        if pd.isna(
            final_row[
                TARGET_COLUMN
            ]
        ):
            continue

        if (
            final_row["target_date"]
            - final_row["date"]
            != pd.Timedelta(days=1)
        ):
            continue

        if (
            final_row["split"]
            not in {
                "train",
                "validation",
                "test",
            }
        ):
            continue

        metadata_rows.append({
            "symbol":
                final_row["symbol"],

            "sequence_start_date":
                dates.iloc[0],

            "sequence_end_date":
                final_row["date"],

            "target_date":
                final_row["target_date"],

            "actual_log_rv":
                float(
                    final_row[
                        TARGET_COLUMN
                    ]
                ),

            "actual_rv":
                float(
                    final_row[
                        ACTUAL_VARIANCE_COLUMN
                    ]
                ),

            "split":
                final_row["split"],
        })

    return pd.DataFrame(
        metadata_rows
    )


metadata_by_lookback = {}

for lookback_days in sorted(
    set(
        MODEL_LOOKBACKS.values()
    )
):
    split_frames = {
        "train": [],
        "validation": [],
        "test": [],
    }

    for symbol in ASSETS:
        symbol_meta = (
            create_asset_sequence_metadata(
                daily_df.loc[
                    daily_df[
                        "symbol"
                    ].eq(symbol)
                ].copy(),
                FEATURE_COLUMNS,
                lookback_days,
            )
        )

        if symbol_meta.empty:
            raise ValueError(
                f"No sequence metadata for "
                f"{symbol}, {lookback_days}D."
            )

        for split in split_frames:
            split_frames[
                split
            ].append(
                symbol_meta.loc[
                    symbol_meta[
                        "split"
                    ].eq(split)
                ].copy()
            )

    metadata_by_lookback[
        lookback_days
    ] = {
        split: (
            pd.concat(
                split_frames[split],
                ignore_index=True,
            )
        )
        for split in split_frames
    }

    print(
        f"\nLookback {lookback_days}D"
    )

    for split in [
        "train",
        "validation",
        "test",
    ]:
        current = (
            metadata_by_lookback[
                lookback_days
            ][split]
        )

        print(
            split,
            len(current),
            current[
                "target_date"
            ].min(),
            current[
                "target_date"
            ].max(),
        )


Lookback 7D
train 8666 2017-09-23 00:00:00+00:00 2024-07-31 00:00:00+00:00
validation 1460 2024-08-01 00:00:00+00:00 2025-07-31 00:00:00+00:00
test 1460 2025-08-01 00:00:00+00:00 2026-07-31 00:00:00+00:00

Lookback 14D
train 8638 2017-09-30 00:00:00+00:00 2024-07-31 00:00:00+00:00
validation 1460 2024-08-01 00:00:00+00:00 2025-07-31 00:00:00+00:00
test 1460 2025-08-01 00:00:00+00:00 2026-07-31 00:00:00+00:00

Lookback 30D
train 8574 2017-10-16 00:00:00+00:00 2024-07-31 00:00:00+00:00
validation 1460 2024-08-01 00:00:00+00:00 2025-07-31 00:00:00+00:00
test 1460 2025-08-01 00:00:00+00:00 2026-07-31 00:00:00+00:00


## 5. Load saved Global DL artifacts and validate metadata alignment

In [5]:
def load_target_scalers(
    artifact_dir,
):
    path = (
        artifact_dir
        / "target_scalers_by_asset.joblib"
    )

    if not path.exists():
        raise FileNotFoundError(
            "Missing target scaler file:\n"
            f"{path}"
        )

    scalers = joblib.load(
        path
    )

    missing_assets = (
        set(ASSETS)
        - set(scalers.keys())
    )

    if missing_assets:
        raise ValueError(
            "Target scalers missing assets: "
            f"{missing_assets}"
        )

    return scalers


def inverse_scaled_log_target(
    scaled_values,
    one_hot,
    target_scalers,
):
    scaled_values = np.asarray(
        scaled_values,
        dtype=float,
    ).reshape(-1)

    one_hot = np.asarray(
        one_hot,
        dtype=float,
    )

    if (
        len(scaled_values)
        != len(one_hot)
    ):
        raise ValueError(
            "Scaled target and one-hot "
            "row counts differ."
        )

    if not np.allclose(
        one_hot.sum(axis=1),
        1.0,
        atol=1e-6,
    ):
        raise ValueError(
            "Asset one-hot rows are invalid."
        )

    output = np.empty(
        len(scaled_values),
        dtype=float,
    )

    asset_index = np.argmax(
        one_hot,
        axis=1,
    )

    for index, symbol in enumerate(
        ASSETS
    ):
        mask = (
            asset_index
            == index
        )

        if not mask.any():
            continue

        output[
            mask
        ] = (
            target_scalers[
                symbol
            ]
            .inverse_transform(
                scaled_values[
                    mask
                ].reshape(
                    -1,
                    1,
                )
            )
            .reshape(-1)
        )

    return output


artifact_data_by_lookback = {}

for lookback_days in sorted(
    set(
        MODEL_LOOKBACKS.values()
    )
):
    artifact_dir = (
        GLOBAL_ARTIFACT_ROOT
        / f"lookback_{lookback_days}d"
    )

    config_path = (
        artifact_dir
        / "experiment_configuration.json"
    )

    if not config_path.exists():
        raise FileNotFoundError(
            "Missing experiment configuration:\n"
            f"{config_path}"
        )

    with open(
        config_path,
        "r",
        encoding="utf-8",
    ) as file:
        config = json.load(
            file
        )

    if (
        int(
            config[
                "lookback_days"
            ]
        )
        != lookback_days
    ):
        raise ValueError(
            "Lookback mismatch in "
            f"{config_path}"
        )

    if (
        config[
            "target"
        ]
        != TARGET_COLUMN
    ):
        raise ValueError(
            "Unexpected target in "
            f"{config_path}: "
            f"{config['target']}"
        )

    if (
        list(
            config[
                "feature_columns"
            ]
        )
        != FEATURE_COLUMNS
    ):
        raise ValueError(
            "Feature list mismatch in "
            f"{config_path}"
        )

    target_scalers = (
        load_target_scalers(
            artifact_dir
        )
    )

    split_data = {}

    for split, file_tag in [
        ("train", "train"),
        ("validation", "val"),
        ("test", "test"),
    ]:
        X_path = (
            artifact_dir
            / f"X_{file_tag}_scaled.npy"
        )

        A_path = (
            artifact_dir
            / (
                f"asset_{file_tag}_"
                "one_hot.npy"
            )
        )

        y_path = (
            artifact_dir
            / f"y_{file_tag}_scaled.npy"
        )

        for path in [
            X_path,
            A_path,
            y_path,
        ]:
            if not path.exists():
                raise FileNotFoundError(
                    f"Missing pooled artifact:\n"
                    f"{path}"
                )

        X = np.load(
            X_path
        )

        A = np.load(
            A_path
        )

        y_scaled = np.load(
            y_path
        ).reshape(-1)

        metadata = (
            metadata_by_lookback[
                lookback_days
            ][split]
            .reset_index(drop=True)
        )

        if not (
            len(X)
            == len(A)
            == len(y_scaled)
            == len(metadata)
        ):
            raise ValueError(
                f"{lookback_days}D {split}: "
                "saved-array and reconstructed-"
                "metadata lengths differ."
            )

        expected_asset_index = np.array(
            [
                ASSETS.index(
                    symbol
                )
                for symbol
                in metadata[
                    "symbol"
                ]
            ],
            dtype=int,
        )

        saved_asset_index = np.argmax(
            A,
            axis=1,
        )

        if not np.array_equal(
            expected_asset_index,
            saved_asset_index,
        ):
            raise ValueError(
                f"{lookback_days}D {split}: "
                "asset one-hot order does not "
                "match reconstructed metadata."
            )

        inverse_y_log = (
            inverse_scaled_log_target(
                y_scaled,
                A,
                target_scalers,
            )
        )

        max_target_diff = float(
            np.max(
                np.abs(
                    inverse_y_log
                    - metadata[
                        "actual_log_rv"
                    ].to_numpy(
                        dtype=float
                    )
                )
            )
        )

        if (
            max_target_diff
            > 1e-5
        ):
            raise ValueError(
                f"{lookback_days}D {split}: "
                "target-scaler alignment failed; "
                f"max difference={max_target_diff}"
            )

        split_data[
            split
        ] = {
            "X": X,
            "A": A,
            "y_scaled": y_scaled,
            "metadata": metadata,
        }

        print(
            f"{lookback_days}D "
            f"{split}: "
            f"{X.shape} | "
            "target alignment max diff="
            f"{max_target_diff:.2e}"
        )

    artifact_data_by_lookback[
        lookback_days
    ] = {
        "artifact_dir":
            artifact_dir,

        "target_scalers":
            target_scalers,

        "splits":
            split_data,
    }

print(
    "\nAll saved Global DL artifacts "
    "passed alignment QC."
)

7D train: (8666, 7, 7) | target alignment max diff=1.33e-06
7D validation: (1460, 7, 7) | target alignment max diff=1.32e-06
7D test: (1460, 7, 7) | target alignment max diff=1.49e-06
14D train: (8638, 14, 7) | target alignment max diff=1.77e-06
14D validation: (1460, 14, 7) | target alignment max diff=1.07e-06
14D test: (1460, 14, 7) | target alignment max diff=9.63e-07
30D train: (8574, 30, 7) | target alignment max diff=1.37e-06
30D validation: (1460, 30, 7) | target alignment max diff=1.09e-06
30D test: (1460, 30, 7) | target alignment max diff=8.95e-07

All saved Global DL artifacts passed alignment QC.


## 6. Load the frozen DL models and generate Train / Validation / Test predictions

`compile=False` is used because no further training is performed.

The final saved model is preferred. If only the best validation checkpoint exists, the notebook uses that checkpoint instead.

In [6]:
def resolve_model_path(
    architecture,
    lookback_days,
):
    model_dir = (
        GLOBAL_MODEL_ROOT
        / f"lookback_{lookback_days}d"
        / architecture
    )

    candidates = [
        model_dir
        / (
            f"final_global_"
            f"{architecture.lower()}.keras"
        ),

        model_dir
        / (
            f"best_global_"
            f"{architecture.lower()}.keras"
        ),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "No saved model file found. Checked:\n"
        + "\n".join(
            str(path)
            for path in candidates
        )
    )


dl_prediction_frames = []
dl_validation_regression_qc = []

for (
    architecture,
    lookback_days,
) in MODEL_LOOKBACKS.items():

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"Global {architecture} "
        f"— {lookback_days}D"
    )

    print(
        "=" * 72
    )

    current = (
        artifact_data_by_lookback[
            lookback_days
        ]
    )

    target_scalers = (
        current[
            "target_scalers"
        ]
    )

    model_path = (
        resolve_model_path(
            architecture,
            lookback_days,
        )
    )

    print(
        "Loading:",
        model_path,
    )

    tf.keras.backend.clear_session()

    model = tf.keras.models.load_model(
        model_path,
        compile=False,
    )

    model_label = (
        f"Global-{architecture}-"
        f"{lookback_days}d"
    )

    for split in [
        "train",
        "validation",
        "test",
    ]:

        split_data = (
            current[
                "splits"
            ][split]
        )

        pred_scaled = (
            model.predict(
                [
                    split_data["X"],
                    split_data["A"],
                ],
                batch_size=32,
                verbose=0,
            )
            .reshape(-1)
        )

        pred_log_rv = (
            inverse_scaled_log_target(
                pred_scaled,
                split_data["A"],
                target_scalers,
            )
        )

        pred_rv = np.exp(
            pred_log_rv
        )

        prediction_df = (
            split_data[
                "metadata"
            ]
            .copy()
        )

        prediction_df[
            "model"
        ] = model_label

        prediction_df[
            "display_model"
        ] = (
            DISPLAY_MODEL_MAP[
                model_label
            ]
        )

        prediction_df[
            "model_family"
        ] = "Deep Learning"

        prediction_df[
            "pred_log_rv"
        ] = pred_log_rv

        prediction_df[
            "pred_rv"
        ] = pred_rv

        dl_prediction_frames.append(
            prediction_df
        )

        print(
            split,
            len(
                prediction_df
            ),
            prediction_df[
                "target_date"
            ].min(),
            "to",
            prediction_df[
                "target_date"
            ].max(),
        )

        if split == "validation":
            saved_prediction_path = (
                GLOBAL_RESULT_ROOT
                / f"lookback_{lookback_days}d"
                / architecture
                / "validation_predictions.parquet"
            )

            if (
                saved_prediction_path.exists()
            ):
                saved = pd.read_parquet(
                    saved_prediction_path
                )

                saved[
                    "target_date"
                ] = pd.to_datetime(
                    saved[
                        "target_date"
                    ],
                    utc=True,
                    errors="raise",
                )

                compare = (
                    prediction_df[
                        [
                            "symbol",
                            "target_date",
                            "pred_rv",
                        ]
                    ]
                    .merge(
                        saved[
                            [
                                "symbol",
                                "target_date",
                                "pred_rv",
                            ]
                        ].rename(
                            columns={
                                "pred_rv":
                                    "saved_pred_rv"
                            }
                        ),
                        on=[
                            "symbol",
                            "target_date",
                        ],
                        how="inner",
                        validate="one_to_one",
                    )
                )

                if (
                    len(compare)
                    != len(
                        prediction_df
                    )
                ):
                    raise ValueError(
                        f"{model_label}: saved "
                        "validation prediction "
                        "coverage mismatch."
                    )

                max_abs_difference = float(
                    np.max(
                        np.abs(
                            compare[
                                "pred_rv"
                            ]
                            - compare[
                                "saved_pred_rv"
                            ]
                        )
                    )
                )

                predictions_match = bool(
                    np.allclose(
                        compare[
                            "pred_rv"
                        ],
                        compare[
                            "saved_pred_rv"
                        ],
                        rtol=1e-3,
                        atol=1e-8,
                    )
                )

                dl_validation_regression_qc.append({
                    "model":
                        model_label,

                    "observations":
                        len(compare),

                    "max_abs_difference":
                        max_abs_difference,

                    "within_tolerance":
                        predictions_match,
                })

                if not predictions_match:
                    raise ValueError(
                        f"{model_label}: newly loaded "
                        "model predictions do not match "
                        "the saved validation forecasts."
                    )

    del model


dl_predictions_df = pd.concat(
    dl_prediction_frames,
    ignore_index=True,
)

if dl_validation_regression_qc:
    display(
        pd.DataFrame(
            dl_validation_regression_qc
        )
    )

print(
    "\nFrozen DL model predictions "
    "generated successfully."
)


Global MLP — 30D
Loading: /content/drive/MyDrive/Quant Research/models/GLOBAL_4_ASSETS/lookback_30d/MLP/final_global_mlp.keras
train 8574 2017-10-16 00:00:00+00:00 to 2024-07-31 00:00:00+00:00
validation 1460 2024-08-01 00:00:00+00:00 to 2025-07-31 00:00:00+00:00
test 1460 2025-08-01 00:00:00+00:00 to 2026-07-31 00:00:00+00:00

Global GRU — 7D
Loading: /content/drive/MyDrive/Quant Research/models/GLOBAL_4_ASSETS/lookback_7d/GRU/final_global_gru.keras
train 8666 2017-09-23 00:00:00+00:00 to 2024-07-31 00:00:00+00:00
validation 1460 2024-08-01 00:00:00+00:00 to 2025-07-31 00:00:00+00:00
test 1460 2025-08-01 00:00:00+00:00 to 2026-07-31 00:00:00+00:00

Global LSTM — 14D
Loading: /content/drive/MyDrive/Quant Research/models/GLOBAL_4_ASSETS/lookback_14d/LSTM/final_global_lstm.keras
train 8638 2017-09-30 00:00:00+00:00 to 2024-07-31 00:00:00+00:00
validation 1460 2024-08-01 00:00:00+00:00 to 2025-07-31 00:00:00+00:00
test 1460 2025-08-01 00:00:00+00:00 to 2026-07-31 00:00:00+00:00


,model,observations,max_abs_difference,within_tolerance
0,Global-MLP-30d,1460,5.292405e-10,True
1,Global-GRU-7d,1460,8.821857e-10,True
2,Global-LSTM-14d,1460,6.187464e-10,True



Frozen DL model predictions generated successfully.


## 7. Determine a common Train evaluation sample

Validation and Test already contain the same 365 dates for all selected DL models.

For Train, the notebook takes the **exact target-date intersection** across MLP-30D, GRU-7D and LSTM-14D separately for each asset. This intersection will also be used for ARCH train errors.

In [7]:
common_train_dates = {}

train_overlap_rows = []

for symbol in ASSETS:

    model_date_sets = []

    for architecture, lookback_days in (
        MODEL_LOOKBACKS.items()
    ):
        model_label = (
            f"Global-{architecture}-"
            f"{lookback_days}d"
        )

        dates = set(
            dl_predictions_df.loc[
                dl_predictions_df[
                    "symbol"
                ].eq(symbol)
                & dl_predictions_df[
                    "split"
                ].eq("train")
                & dl_predictions_df[
                    "model"
                ].eq(
                    model_label
                ),
                "target_date",
            ]
        )

        if not dates:
            raise ValueError(
                f"No train prediction dates for "
                f"{symbol} {model_label}."
            )

        model_date_sets.append(
            dates
        )

    intersection = set.intersection(
        *model_date_sets
    )

    if not intersection:
        raise ValueError(
            f"No common train dates "
            f"for {symbol}."
        )

    common_train_dates[
        symbol
    ] = intersection

    sorted_dates = sorted(
        intersection
    )

    train_overlap_rows.append({
        "asset":
            ASSET_LABELS[
                symbol
            ],

        "common_train_observations":
            len(
                intersection
            ),

        "first_common_train_target":
            sorted_dates[0],

        "last_common_train_target":
            sorted_dates[-1],
    })


common_train_summary_df = (
    pd.DataFrame(
        train_overlap_rows
    )
)

display(
    common_train_summary_df
)

assert (
    common_train_summary_df[
        "last_common_train_target"
    ]
    == TRAIN_END
).all()

print(
    "Common Train evaluation "
    "samples established."
)

,asset,common_train_observations,first_common_train_target,last_common_train_target
0,BTC,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
1,ETH,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
2,SOL,1391,2020-10-10 00:00:00+00:00,2024-07-31 00:00:00+00:00
3,XRP,2221,2018-07-03 00:00:00+00:00,2024-07-31 00:00:00+00:00


Common Train evaluation samples established.


## 8. Reproduce ARCH Train / Validation / Test predictions with frozen Train-estimated parameters

For every asset and ARCH specification:

1. parameters are estimated using Train returns only;
2. Train error uses the fitted in-sample conditional variance;
3. the fitted parameters are then held fixed;
4. realized returns sequentially update the volatility state through Validation and Test;
5. one-step-ahead variance forecasts are evaluated on the canonical target dates.

No model is re-selected and no parameter re-estimation is performed using Validation or Test.

In [8]:
def arch_predictions_for_all_splits(
    symbol,
    model_name,
    specification,
):
    returns = (
        daily_df.loc[
            daily_df[
                "symbol"
            ].eq(symbol),
            [
                "date",
                "daily_return",
            ],
        ]
        .dropna(
            subset=[
                "daily_return"
            ]
        )
        .sort_values("date")
        .drop_duplicates(
            subset=[
                "date"
            ],
            keep="last",
        )
        .set_index("date")[
            "daily_return"
        ]
        .astype(float)
    )

    train_returns = (
        returns.loc[
            returns.index
            <= TRAIN_END
        ]
    )

    validation_returns = (
        returns.loc[
            (
                returns.index
                >= VALIDATION_START
            )
            & (
                returns.index
                <= VALIDATION_END
            )
        ]
    )

    test_returns = (
        returns.loc[
            (
                returns.index
                >= TEST_START
            )
            & (
                returns.index
                <= TEST_END
            )
        ]
    )

    if (
        len(train_returns)
        < MIN_TRAIN_OBS
    ):
        raise ValueError(
            f"{symbol}: insufficient "
            "ARCH training data."
        )

    if len(validation_returns) != 365:
        raise ValueError(
            f"{symbol}: validation returns "
            f"count={len(validation_returns)}."
        )

    if len(test_returns) != 365:
        raise ValueError(
            f"{symbol}: test returns "
            f"count={len(test_returns)}."
        )

    scaled_train = (
        train_returns
        * RETURN_SCALE
    )

    train_model = arch_model(
        scaled_train,
        mean="Constant",
        dist="t",
        rescale=False,
        **specification,
    )

    fit = train_model.fit(
        disp="off",
        show_warning=False,
        update_freq=0,
    )

    if (
        fit.convergence_flag
        != 0
    ):
        warnings.warn(
            f"{symbol} {model_name}: "
            "convergence flag "
            f"{fit.convergence_flag}"
        )

    train_pred_rv = (
        (
            fit.conditional_volatility
            .astype(float)
            ** 2
        )
        / (
            RETURN_SCALE ** 2
        )
    )

    train_prediction_df = (
        pd.DataFrame({
            "symbol":
                symbol,

            "target_date":
                train_pred_rv.index,

            "actual_rv":
                (
                    train_returns
                    .reindex(
                        train_pred_rv.index
                    )
                    .to_numpy(
                        dtype=float
                    )
                    ** 2
                ),

            "pred_rv":
                np.maximum(
                    train_pred_rv
                    .to_numpy(
                        dtype=float
                    ),
                    EPSILON,
                ),

            "split":
                "train",
        })
    )

    # Restrict ARCH train predictions to the same target dates
    # available to all three selected DL models.
    train_prediction_df = (
        train_prediction_df.loc[
            train_prediction_df[
                "target_date"
            ].isin(
                common_train_dates[
                    symbol
                ]
            )
        ]
        .copy()
    )

    full_returns = (
        pd.concat(
            [
                train_returns,
                validation_returns,
                test_returns,
            ]
        )
        .sort_index()
    )

    scaled_full_returns = (
        full_returns
        * RETURN_SCALE
    )

    full_model = arch_model(
        scaled_full_returns,
        mean="Constant",
        dist="t",
        rescale=False,
        **specification,
    )

    fixed_result = (
        full_model.fix(
            fit.params
        )
    )

    forecast = (
        fixed_result.forecast(
            horizon=1,
            start=(
                len(
                    train_returns
                )
                - 1
            ),
            align="target",
            reindex=True,
        )
    )

    forecast_variance = (
        forecast
        .variance[
            "h.1"
        ]
        / (
            RETURN_SCALE ** 2
        )
    )

    out_frames = [
        train_prediction_df
    ]

    for (
        split_name,
        split_label,
        split_returns,
        start_date,
        end_date,
    ) in [
        (
            "validation",
            "Validation",
            validation_returns,
            VALIDATION_START,
            VALIDATION_END,
        ),
        (
            "test",
            "Test",
            test_returns,
            TEST_START,
            TEST_END,
        ),
    ]:

        pred = (
            forecast_variance
            .reindex(
                split_returns.index
            )
        )

        if (
            pred.isna()
            .any()
        ):
            missing_dates = (
                pred.loc[
                    pred.isna()
                ]
                .index
                .tolist()
            )

            raise ValueError(
                f"{symbol} {model_name} "
                f"{split_label}: missing "
                f"forecasts for "
                f"{missing_dates[:5]}"
            )

        frame = pd.DataFrame({
            "symbol":
                symbol,

            "target_date":
                split_returns.index,

            "actual_rv":
                (
                    split_returns
                    .to_numpy(
                        dtype=float
                    )
                    ** 2
                ),

            "pred_rv":
                np.maximum(
                    pred.to_numpy(
                        dtype=float
                    ),
                    EPSILON,
                ),

            "split":
                split_name,
        })

        if (
            frame[
                "target_date"
            ].min()
            != start_date
            or frame[
                "target_date"
            ].max()
            != end_date
            or len(frame)
            != 365
        ):
            raise ValueError(
                f"{symbol} {model_name}: "
                f"{split_label} date QC failed."
            )

        out_frames.append(
            frame
        )

    output = pd.concat(
        out_frames,
        ignore_index=True,
    )

    output[
        "model"
    ] = model_name

    output[
        "display_model"
    ] = (
        DISPLAY_MODEL_MAP[
            model_name
        ]
    )

    output[
        "model_family"
    ] = "ARCH"

    return output


arch_prediction_frames = []

for symbol in ASSETS:

    print(
        "\n"
        + "=" * 72
    )
    print(symbol)
    print("=" * 72)

    for (
        model_name,
        specification,
    ) in ARCH_SPECS.items():

        print(
            "Evaluating",
            model_name,
        )

        arch_prediction_frames.append(
            arch_predictions_for_all_splits(
                symbol=symbol,
                model_name=model_name,
                specification=specification,
            )
        )


arch_predictions_df = pd.concat(
    arch_prediction_frames,
    ignore_index=True,
)

display(
    arch_predictions_df
    .groupby(
        [
            "symbol",
            "model",
            "split",
        ]
    )
    .agg(
        observations=(
            "target_date",
            "size",
        ),
        first_target=(
            "target_date",
            "min",
        ),
        last_target=(
            "target_date",
            "max",
        ),
    )
    .reset_index()
)

print(
    "ARCH Train / Validation / Test "
    "predictions generated successfully."
)


BTCUSDT
Evaluating GARCH(1,1)
Evaluating GJR-GARCH(1,1)
Evaluating EGARCH(1,1)

ETHUSDT
Evaluating GARCH(1,1)
Evaluating GJR-GARCH(1,1)
Evaluating EGARCH(1,1)

SOLUSDT
Evaluating GARCH(1,1)
Evaluating GJR-GARCH(1,1)
Evaluating EGARCH(1,1)

XRPUSDT
Evaluating GARCH(1,1)
Evaluating GJR-GARCH(1,1)
Evaluating EGARCH(1,1)


,symbol,model,split,observations,first_target,last_target
0,BTCUSDT,"EGARCH(1,1)",test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00
1,BTCUSDT,"EGARCH(1,1)",train,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
2,BTCUSDT,"EGARCH(1,1)",validation,365,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00
3,BTCUSDT,"GARCH(1,1)",test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00
4,BTCUSDT,"GARCH(1,1)",train,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
5,BTCUSDT,"GARCH(1,1)",validation,365,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00
6,BTCUSDT,"GJR-GARCH(1,1)",test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00
7,BTCUSDT,"GJR-GARCH(1,1)",train,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
8,BTCUSDT,"GJR-GARCH(1,1)",validation,365,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00
9,ETHUSDT,"EGARCH(1,1)",test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00


ARCH Train / Validation / Test predictions generated successfully.


## 9. Restrict DL Train predictions to the same common dates

Validation and Test are retained in full.

In [9]:
dl_predictions_common_df = (
    dl_predictions_df.copy()
)

train_keep = np.zeros(
    len(
        dl_predictions_common_df
    ),
    dtype=bool,
)

for symbol in ASSETS:

    symbol_train_mask = (
        dl_predictions_common_df[
            "symbol"
        ].eq(symbol)
        & dl_predictions_common_df[
            "split"
        ].eq("train")
    )

    train_keep = (
        train_keep
        | (
            symbol_train_mask
            & dl_predictions_common_df[
                "target_date"
            ].isin(
                common_train_dates[
                    symbol
                ]
            )
        )
    )

non_train_mask = (
    ~dl_predictions_common_df[
        "split"
    ].eq("train")
)

dl_predictions_common_df = (
    dl_predictions_common_df.loc[
        train_keep
        | non_train_mask
    ]
    .copy()
)

dl_predictions_common_df[
    "actual_rv"
] = pd.to_numeric(
    dl_predictions_common_df[
        "actual_rv"
    ],
    errors="raise",
)

dl_predictions_common_df[
    "pred_rv"
] = pd.to_numeric(
    dl_predictions_common_df[
        "pred_rv"
    ],
    errors="raise",
)

display(
    dl_predictions_common_df
    .groupby(
        [
            "symbol",
            "model",
            "split",
        ]
    )
    .agg(
        observations=(
            "target_date",
            "size",
        ),
        first_target=(
            "target_date",
            "min",
        ),
        last_target=(
            "target_date",
            "max",
        ),
    )
    .reset_index()
)

,symbol,model,split,observations,first_target,last_target
0,BTCUSDT,Global-GRU-7d,test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00
1,BTCUSDT,Global-GRU-7d,train,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
2,BTCUSDT,Global-GRU-7d,validation,365,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00
3,BTCUSDT,Global-LSTM-14d,test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00
4,BTCUSDT,Global-LSTM-14d,train,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
5,BTCUSDT,Global-LSTM-14d,validation,365,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00
6,BTCUSDT,Global-MLP-30d,test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00
7,BTCUSDT,Global-MLP-30d,train,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
8,BTCUSDT,Global-MLP-30d,validation,365,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00
9,ETHUSDT,Global-GRU-7d,test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00


## 10. Combine all six models and enforce identical evaluation dates within each asset / split

In [10]:
all_predictions_df = pd.concat(
    [
        arch_predictions_df[
            [
                "symbol",
                "target_date",
                "actual_rv",
                "pred_rv",
                "split",
                "model",
                "display_model",
                "model_family",
            ]
        ],

        dl_predictions_common_df[
            [
                "symbol",
                "target_date",
                "actual_rv",
                "pred_rv",
                "split",
                "model",
                "display_model",
                "model_family",
            ]
        ],
    ],
    ignore_index=True,
)

all_predictions_df[
    "target_date"
] = pd.to_datetime(
    all_predictions_df[
        "target_date"
    ],
    utc=True,
    errors="raise",
)

all_predictions_df[
    "actual_rv"
] = pd.to_numeric(
    all_predictions_df[
        "actual_rv"
    ],
    errors="raise",
)

all_predictions_df[
    "pred_rv"
] = np.maximum(
    pd.to_numeric(
        all_predictions_df[
            "pred_rv"
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=float
    ),
    EPSILON,
)

if all_predictions_df.duplicated(
    subset=[
        "symbol",
        "split",
        "target_date",
        "model",
    ]
).any():
    raise ValueError(
        "Duplicate prediction rows found."
    )


date_qc_rows = []

for symbol in ASSETS:

    for split in [
        "train",
        "validation",
        "test",
    ]:

        current = (
            all_predictions_df.loc[
                all_predictions_df[
                    "symbol"
                ].eq(symbol)
                & all_predictions_df[
                    "split"
                ].eq(split)
            ]
        )

        date_sets = []

        for display_model in (
            MODEL_ORDER
        ):
            dates = set(
                current.loc[
                    current[
                        "display_model"
                    ].eq(
                        display_model
                    ),
                    "target_date",
                ]
            )

            if not dates:
                raise ValueError(
                    f"No dates for "
                    f"{symbol} {split} "
                    f"{display_model}."
                )

            date_sets.append(
                dates
            )

        first_set = (
            date_sets[0]
        )

        if not all(
            dates == first_set
            for dates in date_sets[1:]
        ):
            raise ValueError(
                f"Evaluation date mismatch "
                f"for {symbol} {split}."
            )

        ordered_dates = sorted(
            first_set
        )

        date_qc_rows.append({
            "asset":
                ASSET_LABELS[
                    symbol
                ],

            "split":
                split,

            "observations_per_model":
                len(
                    first_set
                ),

            "first_target":
                ordered_dates[0],

            "last_target":
                ordered_dates[-1],
        })


date_qc_df = pd.DataFrame(
    date_qc_rows
)

display(date_qc_df)

assert (
    date_qc_df.loc[
        date_qc_df[
            "split"
        ].eq("validation"),
        "observations_per_model",
    ]
    == 365
).all()

assert (
    date_qc_df.loc[
        date_qc_df[
            "split"
        ].eq("test"),
        "observations_per_model",
    ]
    == 365
).all()

print(
    "All six models use identical "
    "dates within each asset / split."
)

,asset,split,observations_per_model,first_target,last_target
0,BTC,train,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
1,BTC,validation,365,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00
2,BTC,test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00
3,ETH,train,2481,2017-10-16 00:00:00+00:00,2024-07-31 00:00:00+00:00
4,ETH,validation,365,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00
5,ETH,test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00
6,SOL,train,1391,2020-10-10 00:00:00+00:00,2024-07-31 00:00:00+00:00
7,SOL,validation,365,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00
8,SOL,test,365,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00
9,XRP,train,2221,2018-07-03 00:00:00+00:00,2024-07-31 00:00:00+00:00


All six models use identical dates within each asset / split.


## 11. Calculate Train / Validation / Test MAE, RMSE and QLIKE

In [11]:
def qlike(
    actual_variance,
    predicted_variance,
):
    actual = np.maximum(
        np.asarray(
            actual_variance,
            dtype=float,
        ),
        EPSILON,
    )

    predicted = np.maximum(
        np.asarray(
            predicted_variance,
            dtype=float,
        ),
        EPSILON,
    )

    ratio = (
        actual
        / predicted
    )

    return float(
        np.mean(
            ratio
            - np.log(
                ratio
            )
            - 1.0
        )
    )


metric_rows = []

for (
    symbol,
    split,
    model,
    display_model,
    model_family,
), current in (
    all_predictions_df
    .groupby(
        [
            "symbol",
            "split",
            "model",
            "display_model",
            "model_family",
        ],
        sort=False,
    )
):

    actual = (
        current[
            "actual_rv"
        ]
        .to_numpy(
            dtype=float
        )
    )

    predicted = (
        current[
            "pred_rv"
        ]
        .to_numpy(
            dtype=float
        )
    )

    errors = (
        predicted
        - actual
    )

    metric_rows.append({
        "asset":
            ASSET_LABELS[
                symbol
            ],

        "symbol":
            symbol,

        "split":
            {
                "train":
                    "Train",
                "validation":
                    "Validation",
                "test":
                    "Test",
            }[
                split
            ],

        "model":
            model,

        "display_model":
            display_model,

        "model_family":
            model_family,

        "n_observations":
            int(
                len(current)
            ),

        "mae":
            float(
                np.mean(
                    np.abs(
                        errors
                    )
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    np.mean(
                        errors ** 2
                    )
                )
            ),

        "qlike":
            qlike(
                actual,
                predicted,
            ),
    })


metrics_long_df = (
    pd.DataFrame(
        metric_rows
    )
)

metrics_long_df[
    "asset"
] = pd.Categorical(
    metrics_long_df[
        "asset"
    ],
    categories=ASSET_ORDER,
    ordered=True,
)

metrics_long_df[
    "split"
] = pd.Categorical(
    metrics_long_df[
        "split"
    ],
    categories=SPLIT_ORDER,
    ordered=True,
)

metrics_long_df[
    "display_model"
] = pd.Categorical(
    metrics_long_df[
        "display_model"
    ],
    categories=MODEL_ORDER,
    ordered=True,
)

metrics_long_df = (
    metrics_long_df
    .sort_values(
        [
            "asset",
            "split",
            "display_model",
        ]
    )
    .reset_index(drop=True)
)

display(
    metrics_long_df.style.format({
        "mae":
            "{:.8f}",
        "rmse":
            "{:.8f}",
        "qlike":
            "{:.6f}",
    })
)

,asset,symbol,split,model,display_model,model_family,n_observations,mae,rmse,qlike
0,BTC,BTCUSDT,Train,"GARCH(1,1)","GARCH (1,1)",ARCH,2481,0.00178826,0.00595504,1.982856
1,BTC,BTCUSDT,Train,"GJR-GARCH(1,1)","GJR-GARCH (1,1)",ARCH,2481,0.00177377,0.00593984,1.990385
2,BTC,BTCUSDT,Train,"EGARCH(1,1)","EGARCH (1,1)",ARCH,2481,0.00208786,0.00601963,2.036291
3,BTC,BTCUSDT,Train,Global-MLP-30d,MLP 30D,Deep Learning,2481,0.00136207,0.00670539,4.460179
4,BTC,BTCUSDT,Train,Global-GRU-7d,GRU 7D,Deep Learning,2481,0.00133065,0.00602387,6.685922
5,BTC,BTCUSDT,Train,Global-LSTM-14d,LSTM 14D,Deep Learning,2481,0.00129644,0.00597457,5.944435
6,BTC,BTCUSDT,Validation,"GARCH(1,1)","GARCH (1,1)",ARCH,365,0.00082004,0.00135341,1.865873
7,BTC,BTCUSDT,Validation,"GJR-GARCH(1,1)","GJR-GARCH (1,1)",ARCH,365,0.00081828,0.00135638,1.871826
8,BTC,BTCUSDT,Validation,"EGARCH(1,1)","EGARCH (1,1)",ARCH,365,0.00097331,0.00143365,1.943290
9,BTC,BTCUSDT,Validation,Global-MLP-30d,MLP 30D,Deep Learning,365,0.00057313,0.00140498,3.751286


# Part C — Paper-friendly comparison tables

Each metric is shown in one clean pivot:

- rows = Asset × Split
- columns = Models
- values = corresponding error

This allows Train / Validation / Test performance to be compared without mixing the metrics.

In [12]:
def build_split_metric_pivot(
    metric,
):
    table = (
        metrics_long_df
        .pivot(
            index=[
                "asset",
                "split",
            ],
            columns="display_model",
            values=metric,
        )
        .reindex(
            pd.MultiIndex.from_product(
                [
                    ASSET_ORDER,
                    SPLIT_ORDER,
                ],
                names=[
                    "asset",
                    "split",
                ],
            )
        )
        .reindex(
            columns=MODEL_ORDER
        )
    )

    if table.isna().any().any():
        raise ValueError(
            f"{metric} pivot contains "
            "missing values."
        )

    return table


qlike_table = (
    build_split_metric_pivot(
        "qlike"
    )
)

rmse_table = (
    build_split_metric_pivot(
        "rmse"
    )
)

mae_table = (
    build_split_metric_pivot(
        "mae"
    )
)


print("QLIKE — Train / Validation / Test")
display(
    qlike_table.style.format(
        "{:.6f}"
    )
)

print("RMSE — Train / Validation / Test")
display(
    rmse_table.style.format(
        "{:.8f}"
    )
)

print("MAE — Train / Validation / Test")
display(
    mae_table.style.format(
        "{:.8f}"
    )
)

QLIKE — Train / Validation / Test


RMSE — Train / Validation / Test


MAE — Train / Validation / Test


## 12. Test-only tables

These are convenient for the final held-out-performance section of the paper.

In [13]:
def test_only_table(
    metric,
):
    table = (
        metrics_long_df.loc[
            metrics_long_df[
                "split"
            ].eq("Test")
        ]
        .pivot(
            index="asset",
            columns="display_model",
            values=metric,
        )
        .reindex(
            index=ASSET_ORDER,
            columns=MODEL_ORDER,
        )
    )

    if table.isna().any().any():
        raise ValueError(
            f"Test {metric} table "
            "contains missing values."
        )

    return table


test_qlike_table = (
    test_only_table(
        "qlike"
    )
)

test_rmse_table = (
    test_only_table(
        "rmse"
    )
)

test_mae_table = (
    test_only_table(
        "mae"
    )
)

print("TEST QLIKE")
display(
    test_qlike_table.style.format(
        "{:.6f}"
    )
)

print("TEST RMSE")
display(
    test_rmse_table.style.format(
        "{:.8f}"
    )
)

print("TEST MAE")
display(
    test_mae_table.style.format(
        "{:.8f}"
    )
)

TEST QLIKE


display_model,"GARCH (1,1)","GJR-GARCH (1,1)","EGARCH (1,1)",MLP 30D,GRU 7D,LSTM 14D
asset,,,,,,
BTC,1.605933,1.607774,1.693431,3.730359,3.839056,4.146615
ETH,1.917564,1.919115,1.938653,6.434372,5.206201,5.018030
SOL,1.619505,1.619805,1.593456,3.070938,2.437754,3.405560
XRP,1.859450,1.858839,1.863725,4.795816,4.131717,4.898793


TEST RMSE


display_model,"GARCH (1,1)","GJR-GARCH (1,1)","EGARCH (1,1)",MLP 30D,GRU 7D,LSTM 14D
asset,,,,,,
BTC,0.00153395,0.00153483,0.00157898,0.00156582,0.00156115,0.00156822
ETH,0.00265502,0.00265400,0.00265408,0.00276905,0.00275631,0.00272755
SOL,0.00283853,0.00283806,0.00283547,0.00288011,0.00285682,0.00286604
XRP,0.00383215,0.00382979,0.00372027,0.00377491,0.00374772,0.00374146


TEST MAE


display_model,"GARCH (1,1)","GJR-GARCH (1,1)","EGARCH (1,1)",MLP 30D,GRU 7D,LSTM 14D
asset,,,,,,
BTC,0.00069239,0.00067525,0.00080641,0.00046504,0.00046749,0.00046739
ETH,0.00161857,0.00160347,0.00163284,0.00111849,0.00110795,0.00109572
SOL,0.00191585,0.00191349,0.00187138,0.00122367,0.00121796,0.00122122
XRP,0.00180558,0.00180523,0.00182207,0.00103851,0.00105083,0.00106130


## 13. Save all outputs

In [14]:
LONG_METRICS_PATH = (
    OUTPUT_DIR
    / "all_models_train_validation_test_metrics_long.csv"
)

PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "all_models_train_validation_test_predictions.parquet"
)

DATE_QC_PATH = (
    OUTPUT_DIR
    / "evaluation_date_qc.csv"
)

QLIKE_PATH = (
    OUTPUT_DIR
    / "QLIKE_train_validation_test.csv"
)

RMSE_PATH = (
    OUTPUT_DIR
    / "RMSE_train_validation_test.csv"
)

MAE_PATH = (
    OUTPUT_DIR
    / "MAE_train_validation_test.csv"
)

TEST_QLIKE_PATH = (
    OUTPUT_DIR
    / "TEST_QLIKE.csv"
)

TEST_RMSE_PATH = (
    OUTPUT_DIR
    / "TEST_RMSE.csv"
)

TEST_MAE_PATH = (
    OUTPUT_DIR
    / "TEST_MAE.csv"
)

metrics_long_df.to_csv(
    LONG_METRICS_PATH,
    index=False,
)

all_predictions_df.to_parquet(
    PREDICTIONS_PATH,
    index=False,
)

date_qc_df.to_csv(
    DATE_QC_PATH,
    index=False,
)

qlike_table.to_csv(
    QLIKE_PATH
)

rmse_table.to_csv(
    RMSE_PATH
)

mae_table.to_csv(
    MAE_PATH
)

test_qlike_table.to_csv(
    TEST_QLIKE_PATH
)

test_rmse_table.to_csv(
    TEST_RMSE_PATH
)

test_mae_table.to_csv(
    TEST_MAE_PATH
)

print("Saved:")
for path in [
    LONG_METRICS_PATH,
    PREDICTIONS_PATH,
    DATE_QC_PATH,
    QLIKE_PATH,
    RMSE_PATH,
    MAE_PATH,
    TEST_QLIKE_PATH,
    TEST_RMSE_PATH,
    TEST_MAE_PATH,
]:
    print(" -", path)

Saved:
 - /content/drive/MyDrive/Quant Research/results/FINAL_TRAIN_VALIDATION_TEST_COMPARISON/all_models_train_validation_test_metrics_long.csv
 - /content/drive/MyDrive/Quant Research/results/FINAL_TRAIN_VALIDATION_TEST_COMPARISON/all_models_train_validation_test_predictions.parquet
 - /content/drive/MyDrive/Quant Research/results/FINAL_TRAIN_VALIDATION_TEST_COMPARISON/evaluation_date_qc.csv
 - /content/drive/MyDrive/Quant Research/results/FINAL_TRAIN_VALIDATION_TEST_COMPARISON/QLIKE_train_validation_test.csv
 - /content/drive/MyDrive/Quant Research/results/FINAL_TRAIN_VALIDATION_TEST_COMPARISON/RMSE_train_validation_test.csv
 - /content/drive/MyDrive/Quant Research/results/FINAL_TRAIN_VALIDATION_TEST_COMPARISON/MAE_train_validation_test.csv
 - /content/drive/MyDrive/Quant Research/results/FINAL_TRAIN_VALIDATION_TEST_COMPARISON/TEST_QLIKE.csv
 - /content/drive/MyDrive/Quant Research/results/FINAL_TRAIN_VALIDATION_TEST_COMPARISON/TEST_RMSE.csv
 - /content/drive/MyDrive/Quant Research/

## 14. Final end-to-end QC

In [15]:
# Expected 4 assets × 6 models × 3 splits.
assert (
    len(
        metrics_long_df
    )
    == (
        4
        * 6
        * 3
    )
)

# Exactly six models in every asset / split.
assert (
    metrics_long_df
    .groupby(
        [
            "asset",
            "split",
        ],
        observed=True,
    )[
        "display_model"
    ]
    .nunique()
    .eq(6)
    .all()
)

# Validation and test are exactly 365 observations.
assert (
    metrics_long_df.loc[
        metrics_long_df[
            "split"
        ].eq("Validation"),
        "n_observations",
    ]
    .eq(365)
    .all()
)

assert (
    metrics_long_df.loc[
        metrics_long_df[
            "split"
        ].eq("Test"),
        "n_observations",
    ]
    .eq(365)
    .all()
)

# Train sample size must be identical across all models within an asset.
train_nunique = (
    metrics_long_df.loc[
        metrics_long_df[
            "split"
        ].eq("Train")
    ]
    .groupby(
        "asset",
        observed=True,
    )[
        "n_observations"
    ]
    .nunique()
)

assert (
    train_nunique
    == 1
).all()

# Positive predictions and finite values.
assert (
    all_predictions_df[
        "pred_rv"
    ]
    > 0
).all()

assert np.isfinite(
    all_predictions_df[
        "pred_rv"
    ]
    .to_numpy(
        dtype=float
    )
).all()

assert np.isfinite(
    all_predictions_df[
        "actual_rv"
    ]
    .to_numpy(
        dtype=float
    )
).all()

# All metrics finite and QLIKE non-negative.
assert np.isfinite(
    metrics_long_df[
        [
            "mae",
            "rmse",
            "qlike",
        ]
    ]
    .to_numpy(
        dtype=float
    )
).all()

assert (
    metrics_long_df[
        "qlike"
    ]
    >= -1e-12
).all()

# Canonical validation and test ranges.
validation_rows = (
    all_predictions_df.loc[
        all_predictions_df[
            "split"
        ].eq(
            "validation"
        )
    ]
)

test_rows = (
    all_predictions_df.loc[
        all_predictions_df[
            "split"
        ].eq(
            "test"
        )
    ]
)

assert (
    validation_rows[
        "target_date"
    ].min()
    == VALIDATION_START
)

assert (
    validation_rows[
        "target_date"
    ].max()
    == VALIDATION_END
)

assert (
    test_rows[
        "target_date"
    ].min()
    == TEST_START
)

assert (
    test_rows[
        "target_date"
    ].max()
    == TEST_END
)

# No duplicate model-date observations.
assert not (
    all_predictions_df
    .duplicated(
        subset=[
            "symbol",
            "split",
            "target_date",
            "model",
        ]
    )
    .any()
)

# Pivot dimensions and ordering.
for (
    table_name,
    table,
) in {
    "QLIKE":
        qlike_table,
    "RMSE":
        rmse_table,
    "MAE":
        mae_table,
}.items():

    assert (
        table.shape
        == (
            12,
            6,
        )
    )

    assert (
        list(
            table.columns
        )
        == MODEL_ORDER
    )

    assert not (
        table.isna()
        .any()
        .any()
    )

    print(
        table_name,
        "pivot QC passed."
    )

print(
    "\nALL FINAL TRAIN / VALIDATION / "
    "TEST QC CHECKS PASSED."
)

print(
    "\nIMPORTANT: The held-out Test "
    "period has now been evaluated. "
    "Do not tune models after reading "
    "these Test results."
)

QLIKE pivot QC passed.
RMSE pivot QC passed.
MAE pivot QC passed.

ALL FINAL TRAIN / VALIDATION / TEST QC CHECKS PASSED.

IMPORTANT: The held-out Test period has now been evaluated. Do not tune models after reading these Test results.


# Paper interpretation notes

### Train error
Train errors are in-sample diagnostics. For direct model comparison, the Train table uses the same target dates across all six models within each asset.

### Validation error
Validation performance was used during model/specification selection and therefore should not be presented as a completely untouched final performance estimate.

### Test error
The Test period was held out throughout model development. Once this notebook is run, the Test metrics represent the final independent evaluation and should receive the greatest emphasis in the final results section.

### Recommended reporting
Use QLIKE as the primary volatility-forecasting loss, with MAE and RMSE as complementary metrics. Report Train / Validation / Test together if useful for showing generalization, but base claims about final model performance primarily on the Test results.